In [2]:
from openai import OpenAI

client = OpenAI()

MODEL = "gpt-6-luna"

session = client.beta.agents.sessions.create(
    agent={
        "model": MODEL,
        "instructions": "Answer always in one sentence",
    },
    environment={
        "type": "none",
    },
    input="What is an API?",
)

session.id

'sess_0b9117e786f18da8006ae2c4dff4ec8198a537060f4be8e486'

In [20]:
session.status

'in_progress'

In [21]:
session = client.beta.agents.sessions.retrieve(session.id)

session.to_dict()

{'id': 'sess_00609611b7db8548006ae1c17e94288198b4669523ae97a834',
 'agent': {'id': 'agent_42beb14498514816af3c66f0ff2550ea5d2f967ccfa440ab8a',
  'instructions': 'Answer always in one sentence',
  'model': 'gpt-6-luna',
  'multi_agent': {'enabled': False, 'max_concurrent_subagents': None},
  'name': None,
  'reasoning': {'effort': 'medium', 'summary': None},
  'service_tier': 'auto',
  'text': {'format': {'type': 'text'}, 'verbosity': 'medium'},
  'tools': []},
 'created_at': 1793180030,
 'environment': {'type': 'none'},
 'error': None,
 'last_active_at': 1793180030,
 'metadata': {},
 'object': 'agent.session',
 'required_actions': [],
 'status': 'in_progress',
 'usage': None,
 'vault_ids': []}

In [31]:
items = client.beta.agents.sessions.items.list(session.id, order="asc")

for item in items:
    print(item.to_dict())

{'id': 'msg_888b9f0cdd81e79e6835ee3ff840a07fa7c931e07b48cdd4e4', 'content': [{'text': 'Explain Python Dicts', 'type': 'input_text'}], 'phase': None, 'role': 'user', 'status': 'completed', 'turn_id': 'turn_073dd15352c66ed1006ae1eb701808819b819b12a101e76b92', 'type': 'message'}
{'id': 'msg_076cfd8aaf56303a016ae1eb6f5ee887d081a7069e550c9c64', 'content': [{'text': 'Un **dizionario** Python (`dict`) memorizza coppie **chiave: valore**. È utile per trovare rapidamente un valore usando una chiave descrittiva.\n\n```python\npersona = {\n    "nome": "Luca",\n    "età": 30,\n    "città": "Roma",\n}\n```\n\nPer leggere un valore, usa la chiave:\n\n```python\nprint(persona["nome"])  # Luca\n```\n\nPuoi aggiungere o modificare elementi assegnando un valore alla chiave:\n\n```python\npersona["professione"] = "insegnante"  # aggiunge\npersona["età"] = 31                    # modifica\n```\n\nPer evitare un errore quando la chiave potrebbe mancare, usa `.get()`:\n\n```python\nprint(persona.get("telefo

In [3]:
agent = client.beta.agents.create(
    model=MODEL,
    instructions="You speack only short sentences. Like caveman.",
    reasoning={"effort": "low"},
    text={"verbosity": "low"},
)

agent.id

'agent_57eebc0c0a7b466483de934283b63afd868efe2d998b4372b3'

In [ ]:
with client.beta.agents.sessions.create(
    agent_id=agent.id,
    environment={"type": "none"},
    input="Explain Python Dicts",
    stream=True,
) as events:
    for event in events:
        if event.type == "agent.session.created":
            session_id = event.session.id
        elif event.type == "agent.session.turn.output_text.done":
            print(event.text)
        else:
            print(event.type)

In [1]:
with client.beta.agents.sessions.events.stream(session_id) as events:
    client.beta.agents.sessions.events.create(
        session_id=session_id,
        events=[
            {
                "type": "agent.session.input.message",
                "input": [
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "input_text",
                                "text": "Now explain python tuples.",
                            }
                        ],
                    }
                ],
            }
        ],
    )

    for event in events:
        if event.type == "agent.session.created":
            session_id = event.session.id
        elif event.type == "agent.session.turn.output_text.delta":
            if "```" in event.delta:
                client.beta.agents.sessions.events.create(
                    session_id,
                    events=[
                        {
                            "type": "agent.session.input.cancel",
                        }
                    ],
                )
        elif event.type == "agent.session.turn.output_text.done":
            print(event.text)
        elif event.type != "agent.session.turn.output_text.delta":
            print(event.type)

NameError: name 'client' is not defined

In [25]:
client.beta.agents.update(
    agent_id=agent.id,
    instructions="You reply only in Italian.",
)

Agent(id='agent_4d1cb32766f0429895b1ed92e171e86b77ab1b5b5b88401aa8', created_at=1793180033, instructions='You reply only in Italian.', metadata={}, model='gpt-6-luna', multi_agent=MultiAgentConfig(enabled=False, max_concurrent_subagents=None), name=None, object='agent', reasoning=AgentReasoning(effort='low', summary=None), service_tier='auto', text=AgentText(format=TextFormatResourceText(type='text'), verbosity='low'), tools=[], updated_at=1793180034)

In [27]:
client.beta.agents.sessions.events.create(
    session_id=session.id,
    events=[
        {
            "type": "agent.session.input.message",
            "input": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "input_text",
                            "text": "Now explain async",
                        }
                    ],
                },
            ],
        }
    ],
)

In [30]:
items = client.beta.agents.sessions.items.list(
    session_id=session.id,
    order="asc",
)

for item in items:
    print(item.to_dict())

{'id': 'msg_888b9f0cdd81e79e6835ee3ff840a07fa7c931e07b48cdd4e4', 'content': [{'text': 'Explain Python Dicts', 'type': 'input_text'}], 'phase': None, 'role': 'user', 'status': 'completed', 'turn_id': 'turn_073dd15352c66ed1006ae1eb701808819b819b12a101e76b92', 'type': 'message'}
{'id': 'msg_076cfd8aaf56303a016ae1eb6f5ee887d081a7069e550c9c64', 'content': [{'text': 'Un **dizionario** Python (`dict`) memorizza coppie **chiave: valore**. È utile per trovare rapidamente un valore usando una chiave descrittiva.\n\n```python\npersona = {\n    "nome": "Luca",\n    "età": 30,\n    "città": "Roma",\n}\n```\n\nPer leggere un valore, usa la chiave:\n\n```python\nprint(persona["nome"])  # Luca\n```\n\nPuoi aggiungere o modificare elementi assegnando un valore alla chiave:\n\n```python\npersona["professione"] = "insegnante"  # aggiunge\npersona["età"] = 31                    # modifica\n```\n\nPer evitare un errore quando la chiave potrebbe mancare, usa `.get()`:\n\n```python\nprint(persona.get("telefo